In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



Zad 1

In [2]:
def distp(X, C):
    """
    Oblicza odległość euklidesową między dwoma zbiorami punktów X i C.
    
    Parametry:
    X - macierz n x m (n próbek, m cech)
    C - macierz k x m (k centroidów, m cech)
    
    Zwraca:
    d - macierz n x k odległości między każdą próbką a każdym centroidem
    """
    # Rozszerzamy wymiary, aby umożliwić broadcasting
    X_expanded = X[:, np.newaxis, :]  # kształt (n, 1, m)
    C_expanded = C[np.newaxis, :, :]  # kształt (1, k, m)
    
    # Obliczamy różnice i podnosimy do kwadratu
    squared_diff = np.square(X_expanded - C_expanded)
    
    # Sumujemy po wymiarze cech (ostatni wymiar) i pierwiastkujemy
    distances = np.sqrt(np.sum(squared_diff, axis=2))
    
    return distances

In [3]:
def distm(X, C, V=None):
    """
    Oblicza odległość Mahalanobisa między dwoma zbiorami punktów X i C.
    
    Parametry:
    X - macierz n x m (n próbek, m cech)
    C - macierz k x m (k centroidów, m cech)
    V - opcjonalna macierz kowariancji (m x m), jeśli None, obliczana z X
    
    Zwraca:
    d - macierz n x k odległości Mahalanobisa
    """
    if V is None:
        V = np.cov(X.T)  # Obliczamy macierz kowariancji, jeśli nie podana
    
    try:
        V_inv = np.linalg.inv(V)  # Odwrotność macierzy kowariancji
    except np.linalg.LinAlgError:
        # Jeśli macierz jest osobliwa, dodajemy małą wartość do diagonali
        V_inv = np.linalg.pinv(V)
    
    # Rozszerzamy wymiary dla broadcasting
    X_expanded = X[:, np.newaxis, :]  # kształt (n, 1, m)
    C_expanded = C[np.newaxis, :, :]  # kształt (1, k, m)
    
    # Różnica między punktami a centroidami
    diff = X_expanded - C_expanded  # kształt (n, k, m)
    
    # Obliczanie odległości Mahalanobisa
    distances = np.sqrt(np.einsum('nkm,ml,nkl->nk', diff, V_inv, diff))
    
    return distances

In [ ]:
def ksrodki(X, k, max_iter=100, tol=1e-4, metric='euclidean'):
    """
    Implementacja algorytmu k-środków.
    
    Parametry:
    X - macierz danych n x m
    k - liczba klastrów
    max_iter - maksymalna liczba iteracji
    tol - tolerancja zmiany centroidów
    metric - 'euclidean' lub 'mahalanobis'
    
    Zwraca:
    C - macierz centroidów k x m
    CX - wektor przynależności n x 1
    """
    n, m = X.shape
    
    # 1. Inicjalizacja centroidów - wybieramy losowe punkty z danych
    random_indices = np.random.choice(n, size=k, replace=False)
    C = X[random_indices, :]
    
    # Inicjalizacja wektora przynależności
    CX = np.zeros(n, dtype=int)
    
    for iteration in range(max_iter):
        # 2. Przypisanie punktów do najbliższych centroidów
        if metric == 'euclidean':
            distances = distp(X, C)
        elif metric == 'mahalanobis':
            distances = distm(X, C)
        else:
            raise ValueError("Nieznana metryka")
            
        new_CX = np.argmin(distances, axis=1)
        
        # 3. Sprawdzenie warunku stopu
        if np.all(new_CX == CX):
            break
            
        CX = new_CX
        
        # 4. Aktualizacja centroidów
        for i in range(k):
            # Wybieramy punkty należące do i-tego klastra
            cluster_points = X[CX == i, :]
            if len(cluster_points) > 0:
                C[i, :] = np.mean(cluster_points, axis=0)
    
    return C, CX